In [ ]:
import ee
import geemap
from utils import *

initialize()
config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

In [74]:
age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020").rename("age")

biomass = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').mean().select("AGB").rename("biomass")

sd = ee.ImageCollection("projects/sat-io/open-datasets/ESA/ESA_CCI_AGB").filterDate('2020-01-01','2021-01-01').first().select("SD").rename("sd")

amazon = ee.FeatureCollection("projects/extents-490617/assets/biomes_br").filter(ee.Filter.eq('Bioma', 'Amazônia'))


## Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

In [ ]:

grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    task.start()

#309 needs to be run again


# Get biomass and age for each secondary forest patch

In [ ]:

def import_folder_features(folder_path):
    # 1. List all assets in the folder
    # returns a list of dictionaries with 'name', 'type', and 'id'
    asset_list = ee.data.listAssets({'parent': folder_path})['assets']
    
    # 2. Filter for 'TABLE' assets (FeatureCollections)
    # This prevents errors if you have Images in the same folder
    feature_ids = [a['name'] for a in asset_list if a['type'] == 'TABLE']
    
    # 3. Load each ID into an ee.FeatureCollection
    # We use a standard list comprehension here
    collections = [ee.FeatureCollection(asset_id) for asset_id in feature_ids]
    
    print(f"Found {len(collections)} FeatureCollections.")
    return collections

# Usage
my_folder = 'projects/my-project/assets/my_shapefiles'
all_features = import_folder_features(f"{data_folder}/secondary_polygons")

# Optional: Merge them all into one single FeatureCollection
if all_features:
    merged_collection = ee.FeatureCollection(all_features).flatten()

# map = geemap.Map()
# map.addLayer(merged_collection, {}, "Merged Secondary Polygons")
# map


# get one biomass value per polygon


Found 22 FeatureCollections.


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [ ]:
# map.addLayer(biomass, {}, "biomass") #'min':30, 'max':150, 'palette': ['lightgreen', 'darkgreen']

# map.addLayer(age, {'min':0, 'max':35, 'palette': ['yellow', 'orange', 'red']}, "age")